In [39]:
from dotenv import load_dotenv
load_dotenv()

True

In [40]:
from groq import Groq

client = Groq()



In [41]:
def llm(prompt):
    model = client.chat.completions.create(
        
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
    )
    return model.choices[0].message.content

In [42]:
print(llm("What is the capital of France?"))

The capital of France is Paris.


In [43]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"

response = requests.get(docs_url)

print(response.status_code) 

200


In [6]:
courses = response.json()

In [7]:
documents = []

prefix_url = "https://datatalks.club/faq/"

for course in courses:
    url = f"{prefix_url}{course['path']}"
    
    course_response = requests.get(url)
    course_response.raise_for_status()
    course_data = course_response.json()
    
    documents.extend(course_data)

In [8]:
len(documents)

1342

In [9]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [10]:
documents_cleaned = []

for doc in documents:
    documents_cleaned.append({
        "course": doc['course'],
        'section': doc['section'],
        'question': doc['question'],
        'answer': doc['answer']
                      })
    
documents_cleaned[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [11]:
from minsearch import Index

In [13]:
index = Index(
    text_fields = ['question', 'answer', 'section'],
    keyword_fields = ['course']
)

index.fit(documents_cleaned)

In [14]:
def search_db(question, course):
    search_results = index.search(
        question,
        boost_dict={"question": 2.0, 'section': 0.5},
        filter_dict={"course": course},
        num_results=5
        )
    return search_results

In [28]:
question = "I just discovered the course. Can I join now?"

course = 'llm-zoomcamp'

search_db(question, course)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" c

In [24]:
def format_context(documents):
    context = []
    for doc in documents:
        context.append(f"""
        Question: {doc['question']}
        Answer: {doc['answer']}
        section: {doc['section']}
        """
        )
    return "\n\n".join(context)

In [25]:
def build_prompt(question, context):
    prompt = f"""
    Your task is to answer questions from the course participants
    based on the provided context.

    Use the context to find relevant information and provide accurate
    answers. If the answer is not found in the context,
    respond with "I don't know."

    Question:
    {question}

    Context:
    {context}
    """
    return prompt

In [32]:
def ask(question,course):
    docs = search_db(question,course)
    context = format_context(docs)
    prompt = build_prompt(question, context)
    answer = llm(prompt)
    
    return answer


In [33]:
print(ask(question,course))

Yes, you can join now, but if you want to receive a certificate, you need to submit your project while they're still accepting submissions.


In [34]:
from rag_helper import ask

question = "Is there a certificate?"
course = "machine-learning-zoomcamp"

answer = ask(question, course)
print(answer)


    Your task is to answer questions from the course participants
    based on the provided context.

    Use the context to find relevant information and provide accurate
    answers. If the answer is not found in the context,
    respond with "I don't know."

    Question:
    Is there a certificate?

    Context:
    
        Question: Will I get a certificate?
        Answer: Yes, if you finish at least 2 out of 3 projects and review 3 peers’ projects by the deadline, you will get a certificate. This is what it looks like: [this](https://certificate.datatalks.club/mlzoomcamp/2021/35fc7e051003fddcc6909a8ee96703bd9c31b454.pdf).
        section: General Course-Related Questions
        


        Question: What do I need to earn the certificate?
        Answer: - You must pass 2 out of the 3 projects to earn the certificate: Midterm + Capstone 1, or Capstone 1 + Capstone 2.
- Certificates show pass/fail only—no percentage or rank.
- Eligibility for the certificate also follows the ex

In [49]:
from ingest import load_faqs, build_index
from rag_helper import RAGBase


document = load_faqs()
index = build_index(document)
rag = RAGBase(index, client)

print(rag.rag("Is there a certificate?", "machine-learning-zoomcamp"))

ImportError: cannot import name 'load_faqs' from 'ingest' (/workspaces/llm-zoomcamp-code/Module 1/ingest.py)